# Fine-tune Qwen2.5-Coder for React Test Generation

This notebook fine-tunes `qwen2.5-coder:3b` on your expense-manager React test dataset.

**Prerequisites:**
1. Upload `dataset.jsonl` to this Colab (use the file browser on the left)
2. Make sure GPU is enabled: **Runtime → Change runtime type → T4 GPU**
3. Run all cells in order (~30-60 min total)
4. Download the exported `.gguf` file at the end

## Step 1: Install Dependencies (~3 min)

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes

## Step 2: Load Model with 4-bit Quantization

In [ ]:
from unsloth import FastLanguageModel
import torch
import json

MODEL_NAME = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 4096
LORA_RANK = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Add LoRA adapters (only trains ~1-2% of parameters)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"Trainable parameters: {model.print_trainable_parameters()}")

## Step 3: Upload & Load Training Data

⬆️ **Upload `dataset.jsonl` now** using the file browser on the left (folder icon), then run this cell.

In [ ]:
def load_dataset_jsonl(path):
    """Load the JSONL dataset produced by build-training-data.mjs"""
    examples = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data = json.loads(line)
            examples.append(data)
    return examples

DATASET_PATH = "dataset.jsonl"
raw_data = load_dataset_jsonl(DATASET_PATH)
print(f"Loaded {len(raw_data)} training examples")

# Format for Qwen chat template
def format_for_training(example):
    messages = example["messages"]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

from datasets import Dataset

dataset = Dataset.from_list([format_for_training(ex) for ex in raw_data])
print(f"Dataset ready: {len(dataset)} examples")
print(f"\nSample (first 300 chars):\n{dataset[0]['text'][:300]}")

## Step 4: Configure Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=5,
        lr_scheduler_type="linear",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        logging_steps=1,
        output_dir="outputs",
        save_strategy="epoch",
        seed=42,
        gradient_checkpointing=True,
    ),
)

## Step 5: Train! (~30-60 min)

In [ ]:
print("Starting fine-tuning...")
trainer_stats = trainer.train()

print(f"\n✅ Training complete!")
print(f"  Loss: {trainer_stats.training_loss:.4f}")
print(f"  Runtime: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"  Samples/sec: {trainer_stats.metrics['train_samples_per_second']:.2f}")

## Step 6: Test the Fine-tuned Model

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = """Generate a test file for this React component:

## File: components/common/Button.tsx
```tsx
import React from 'react';

interface ButtonProps {
  label: string;
  onClick: () => void;
  disabled?: boolean;
  variant?: 'primary' | 'secondary';
}

export const Button: React.FC<ButtonProps> = ({ label, onClick, disabled, variant = 'primary' }) => {
  return (
    <button className={`btn btn-${variant}`} onClick={onClick} disabled={disabled}>
      {label}
    </button>
  );
};
```"""

messages = [
    {"role": "system", "content": "You are a React testing expert. Generate comprehensive Jest + RTL tests."},
    {"role": "user", "content": test_prompt},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=1024,
    temperature=0.2,
    top_p=0.9,
)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("Generated test:\n")
print(response)

## Step 7: Export to GGUF (for Ollama)

This creates a `.gguf` file you can import into Ollama on your local machine.

In [ ]:
print("Exporting to GGUF format (Q4_K_M quantization)...")
model.save_pretrained_gguf(
    "testgen-coder-finetuned",
    tokenizer,
    quantization_method="q4_k_m",
)
print("\n✅ GGUF export complete!")

## Step 8: Download the Model

Run this cell to download the `.gguf` file to your computer.

In [ ]:
from google.colab import files
import os

gguf_path = "testgen-coder-finetuned/unsloth.Q4_K_M.gguf"
size_mb = os.path.getsize(gguf_path) / (1024 * 1024)
print(f"Model size: {size_mb:.0f} MB")
print(f"Downloading {gguf_path}...")
files.download(gguf_path)

## After Download: Import into Ollama

On your local machine (Windows), run these commands:

```bash
# 1. Create a Modelfile (in the same folder as the downloaded .gguf)
echo 'FROM ./unsloth.Q4_K_M.gguf' > Modelfile

# 2. Import into Ollama
ollama create testgen-coder-finetuned -f Modelfile

# 3. Test it
ollama run testgen-coder-finetuned

# 4. Use with testgen
set OLLAMA_MODEL=testgen-coder-finetuned
npm run testgen:enhance
```